In [7]:
import pandas as pd
import sys
sys.path.append("..")
from config import REGION_PROCESSED_DIR
from sklearn.model_selection import train_test_split   

# read the training table
train_df = pd.read_csv(REGION_PROCESSED_DIR / "training_table.csv")

# 1. correlation with the target
cols = ["past_crashes", "past_injured", "past_killed", "n_segments", "pci", "target_crashes"]
print(train_df[cols].corr()["target_crashes"])

# 2.  split 80% train, 20% test
train_set, test_set = train_test_split(train_df, test_size=0.2, random_state=42)
print(len(train_set), len(test_set))

past_crashes      0.780696
past_injured      0.763404
past_killed       0.271640
n_segments        0.420101
pci               0.066433
target_crashes    1.000000
Name: target_crashes, dtype: float64
6577 1645


In [8]:
from sklearn.metrics import mean_absolute_error

# independent copy of the test set
test_set = test_set.copy()

# ---------- Prediction 1: past rate ----------
test_set["pred_rate"] = test_set["past_crashes"] * 3 / 18

mean_value = train_set["target_crashes"].mean()
test_set["pred_mean"] = mean_value

# ---------- MAE for each prediction ----------
y_true = test_set["target_crashes"]
print("MAE rate:", mean_absolute_error(y_true, test_set["pred_rate"]))
print("MAE mean:", mean_absolute_error(y_true, test_set["pred_mean"]))

# ---------- Top 10% coverage ----------
n = int(len(test_set) * 0.1)                       # size of top 10%
total_crashes = y_true.sum()                       # all test crashes

top_pred = test_set.sort_values("pred_rate", ascending=False).head(n)
coverage = top_pred["target_crashes"].sum() / total_crashes

top_ideal = test_set.sort_values("target_crashes", ascending=False).head(n)
ideal = top_ideal["target_crashes"].sum() / total_crashes

print("Top 10% coverage (model):", coverage)
print("Top 10% coverage (ideal):", ideal)

MAE rate: 0.5829787234042553
MAE mean: 1.0508378419221815
Top 10% coverage (model): 0.5286984640258691
Top 10% coverage (ideal): 0.6523848019401779


In [9]:
print("MAE zero:", mean_absolute_error(y_true, [0] * len(y_true)))

MAE zero: 0.7519756838905776


In [10]:
import xgboost as xgb
#from xgboost import XGBRegressor    
from sklearn.metrics import mean_absolute_error
features = ["past_crashes", "past_injured", "past_killed", "n_segments", "main_class", "pci"]
X_train = train_set[features]
Y_train = train_set["target_crashes"]
X_test = test_set[features]
Y_test = test_set["target_crashes"]
model = xgb.XGBRegressor(objective="count:poisson", n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)

model.fit(X_train, Y_train)
test_set["pred_xgb"] = model.predict(X_test)    
MAE_xgb = mean_absolute_error(Y_test, test_set["pred_xgb"])
print("MAE XGB:", MAE_xgb)  


MAE XGB: 0.5996918678283691


In [11]:

def top_coverage(df, pred_col, frac=0.1, target_col="target_crashes"):
    #Share of actual crashes captured by the top-frac intersections by prediction.
    n = int(len(df) * frac) #  size of top group
    top = df.sort_values(pred_col, ascending=False).head(n) #  highest predictions
    return top[target_col].sum() / df[target_col].sum() # their actual crashes ÷ total

In [12]:
print("MAE XGB:", mean_absolute_error(Y_test, test_set["pred_xgb"]))
print("Coverage XGB :", top_coverage(test_set, "pred_xgb"))
print("Coverage Rate:", top_coverage(test_set, "pred_rate"))
print("Coverage Ideal:", top_coverage(test_set, "target_crashes"))

MAE XGB: 0.5996918678283691
Coverage XGB : 0.5303152789005658
Coverage Rate: 0.5286984640258691
Coverage Ideal: 0.6523848019401779


In [13]:
test_set = test_set.copy()
test_set["pred_rate5"] = test_set["past5_crashes"] * 3 / 5
test_set["pred_rate2"] = test_set["past2_crashes"] * 3 / 2

#  new features for XGBoost
features2 = ["past_crashes", "past5_crashes", "past2_crashes",
             "past_injured", "past_killed", "n_segments", "main_class", "pci"]

model2 = xgb.XGBRegressor(objective="count:poisson", n_estimators=300,
                          max_depth=3, learning_rate=0.05, random_state=42)
model2.fit(train_set[features2], train_set["target_crashes"])
test_set["pred_xgb2"] = model2.predict(test_set[features2])

# compare all predictions
for col in ["pred_rate", "pred_rate5", "pred_rate2", "pred_xgb", "pred_xgb2"]:
    mae = mean_absolute_error(test_set["target_crashes"], test_set[col])
    print(f"{col:12s} MAE={mae:.3f} coverage={top_coverage(test_set, col):.3f}")


pred_rate    MAE=0.583 coverage=0.529
pred_rate5   MAE=0.604 coverage=0.513
pred_rate2   MAE=0.637 coverage=0.494
pred_xgb     MAE=0.600 coverage=0.530
pred_xgb2    MAE=0.591 coverage=0.541
